# LASSER-VAE: a VAE-learned co-expression graph, where the graph can actually act

**Background (from the previous studies).** In GEARS the co-expression GNN only smooths a learnable per-gene table
that is the same for every cell, so *any* graph only re-parameterises per-gene biases. Learned, Pearson and random
graphs gave the same predictions. This study changes two things:

1. **How the graph is learned (`graph_mode="vae"`).** A cell-level VAE with a linear decoder is trained on training
   cells (half H1). Its decoder loadings are gene embeddings. Three gene-gene similarities are combined:
   - `load`: shared programs (decoder loadings);
   - `denoise`: correlation of the VAE's reconstructed, de-noised expression;
   - `resp`: correlation of decoded responses across training perturbations.

   Edge confidence comes from posterior sampling. The graph is refined from GEARS's A0 exactly like LASSER-GSR.
2. **Where the graph acts (`cell_signal=True`, opt-in, implies the batching fix).** Each cell's control expression
   is fed into the co-expression GNN, so graph messages differ by cell. Now graph structure *can* matter.

| Tier | Jobs (10 epochs each) | Question |
|---|---|---|
| 1 | `GEARS-cs` vs `VAE-cs`, splits 1–3 | Does the VAE graph beat Pearson once the graph acts on cell signal? |
| 2 | `VAE-random-cs`, splits 1–3 | Learned partners, or just extra edges? (same in-degrees and weights) |
| 3 | `GEARS-kNN8-cs`, splits 1–3 | More than a denser Pearson graph? |
| 4 | `LASSER-cs`, splits 1–3 | VAE graph vs GSR graph |
| 5 | `VAE-fix`, splits 1–3 | VAE graph in GEARS's original slot (expected: no effect) |
| 6 | training-seed replicates of tier 1 | Run-to-run noise |
| 7 | splits 4–5 | More splits |

**Budget.** About 50 min per run on one GPU, so one **10-hour** session finishes tiers 1–2 and most of tier 3
(about 10–11 runs). Jobs run in priority order, interleaved by split, so whatever finishes is a complete paired
set. Later sessions resume where this one stopped.

**Kaggle settings:** GPU (one T4 is enough; RAM allows only one worker), **Internet on**. Run with
*Save Version → Save & Run All (Commit)*. To resume, add this notebook's previous output as input and set
`RESUME_FROM`.

Results are compared only within this study (10 epochs, same variant), not with earlier studies.

## A. Setup and experiments

In [ ]:
# 1. Dependencies (GEARS is vendored inside lasser/).
!pip install -q torch_geometric scanpy pytest

In [ ]:
# 2. Code.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# 3. Settings.
import os, shutil, time
T0 = time.time()
SESSION_HOURS = 10.0                # your limit for this session
ANALYSIS_RESERVE_HOURS = 0.6        # kept free at the end for analysis + saving outputs
EPOCHS = 10
STUDY_DIR = "/kaggle/working/vae_study"
DATA_DIR = "/tmp/lasser_data"       # not saved as output (several GB); re-downloaded each session
RESUME_FROM = None                  # e.g. "/kaggle/input/<previous-version>/vae_study"
RUN_DATA_TESTS = False              # True: also the dataset tests incl. the VAE leakage test (~25 min)

os.makedirs(STUDY_DIR, exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, STUDY_DIR, dirs_exist_ok=True)
    shutil.rmtree(f"{STUDY_DIR}/runs/claims", ignore_errors=True)
    print("resumed from", RESUME_FROM)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!free -g | head -2

In [ ]:
# 4. Fast tests (seconds, no data). test_08 covers the VAE, the similarity identities, the cell-signal variant
#    and the config/hash compatibility (earlier GSR runs and caches stay valid).
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_05_parsing.py {REPO_DIR}/tests/test_06_study.py {REPO_DIR}/tests/test_07_fix.py {REPO_DIR}/tests/test_08_vae.py -q -p no:cacheprovider --tb=short 2>&1 | grep -vE "Warning|warn|^  |^$"
if RUN_DATA_TESTS:
    %env LASSER_DATA_DIR={DATA_DIR}
    %env LASSER_TEST_DIR=/tmp/lasser_tests
    !cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -q -p no:cacheprovider --tb=short 2>&1 | tail -n 30

In [ ]:
# 5. Experiment table (priority order) and what is already done.
import pandas as pd
from lasser.study import ARM_DESCRIPTIONS, VAE_PLAN, make_jobs, study_status, expected_minutes, default_minutes

BASE = dict(dataset="norman", split="simulation", epochs=EPOCHS, device="cuda",
            data_dir=DATA_DIR, out_dir=f"{STUDY_DIR}/runs", cache_dir=f"{STUDY_DIR}/cache", repo_dir=REPO_DIR)
jobs = make_jobs(VAE_PLAN, BASE)
status = study_status(jobs)
est = expected_minutes(BASE["out_dir"], default_minutes(EPOCHS))
left = SESSION_HOURS - ANALYSIS_RESERVE_HOURS - (time.time() - T0) / 3600
print(f"{len(jobs)} jobs, {int(status.done.sum())} done; ~{est:.0f} min each -> ~{int(left * 60 // est)} fit in this session")
display(status[["job_id", "arm", "seed", "train_seed", "done"]].assign(
    what=lambda d: d.arm.map(ARM_DESCRIPTIONS)).head(40))

In [ ]:
# 6. Data, splits and GEARS graphs, built once (~15 min the first time).
from lasser.study import prepare_data
if not study_status(jobs).done.all():
    prepare_data(jobs)

In [ ]:
# 7. Run jobs in priority order until the budget is used (one worker; progress every 10 min).
from lasser.study import launch
budget = SESSION_HOURS - ANALYSIS_RESERVE_HOURS - (time.time() - T0) / 3600
if not study_status(jobs).done.all():
    launch(jobs, budget_hours=budget, poll_minutes=10)
status = study_status(jobs)
print(f"{int(status.done.sum())}/{len(status)} jobs done")
if os.path.exists(f"{BASE['out_dir']}/failures.csv"):
    display(pd.read_csv(f"{BASE['out_dir']}/failures.csv").tail(5))
status[status.done][["job_id", "test_pearson_delta", "train_minutes"]]

## B. Analysis

Works on whatever has finished, and can be rerun alone (CPU) on a previous output by pointing `STUDY_DIR` at it.

- The **split is the unit of replication**: training-seed replicates are averaged within a split
  (`unit="split"`). `unit="run"` pairs every (split, training seed).
- Arms with the same split and training seed share initialisation and batch order, so their difference isolates
  the graph.
- Improvement > 0 means better, for every metric.

In [ ]:
# 8. Load runs and the reference baselines (no perturbation, additive).
import numpy as np
from lasser import analysis as A
from lasser import figures as F
F.use_vae_colors()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
RESULTS_DIR, FIG_DIR = f"{STUDY_DIR}/results", f"{STUDY_DIR}/figures"
runs = A.load_runs(f"{STUDY_DIR}/runs")
long = A.metrics_long(runs)
H5AD = f"{DATA_DIR}/norman/perturb_processed.h5ad"
if os.path.exists(H5AD):
    long = pd.concat([long, A.paper_reference_baselines(runs, H5AD)], ignore_index=True)
ARMS = [a for a in ["GEARS-cs", "VAE-cs", "GEARS-kNN8-cs", "VAE-random-cs", "LASSER-cs", "VAE-fix"]
        if a in set(runs.arm)]
REFS = [a for a in ["No-perturb", "Additive"] if a in set(long[long.family == "paper"].arm)]
SCOPES = ["overall"] + A.SUBGROUPS
display(runs.pivot_table(index="arm", columns=["seed", "rep"], values="test_pearson_delta").reindex(ARMS).round(3))

In [ ]:
# 9. The VAE itself: training, held-out reconstruction, latent usage, edge confidence, graph checks.
cols = [c for c in ["arm", "seed", "vae_heldout_r2", "vae_active_dims", "vae_edge_confidence", "graph_h2_hit_added",
                    "graph_h2_hit_random", "graph_num_edges", "graph_isolated_genes", "graph_degree_median"]
        if c in runs.columns]
display(runs[runs.arm.str.startswith("VAE")][cols].drop_duplicates(["arm", "seed"]).round(3))

In [ ]:
# 10. Table 1: headline metrics (mean ± sd over splits; replicates averaged within split).
t1 = A.summary_table(long, A.HEADLINE, scopes=["overall"], arms=REFS + ARMS)
t1_sub = A.summary_table(long, ["pearson_delta", "mse_top20_de", "frac_opposite_top20_de"], scopes=A.SUBGROUPS,
                         arms=REFS + ARMS)
display(t1); display(t1_sub)

In [ ]:
# 11. Table 2: the paired comparisons that carry the claim.
COMPARISONS = [("GEARS-cs", "VAE-cs"),          # VAE graph vs Pearson graph (cell-signal variant)
               ("VAE-random-cs", "VAE-cs"),     # learned vs random partners, same degrees/weights
               ("GEARS-kNN8-cs", "VAE-cs"),     # VAE vs density-matched Pearson
               ("LASSER-cs", "VAE-cs"),         # VAE vs GSR graph
               ("GEARS-cs", "GEARS-kNN8-cs")]   # does a denser Pearson graph help in this variant?
show = ["arm", "ref", "unit", "scope", "metric", "n_splits", "ref_mean", "arm_mean", "improvement", "ci95_low",
        "ci95_high", "splits_better", "p_t", "p_wilcoxon"]
t2 = pd.concat([A.paired_tests(long, ref, [arm], A.HEADLINE, scopes=SCOPES, unit=u)
                for ref, arm in COMPARISONS if {ref, arm} <= set(ARMS) for u in ("split", "run")], ignore_index=True)
if len(t2):
    t2["p_t_holm"] = np.nan
    for u in ("split", "run"):
        m = t2.unit == u
        t2.loc[m, "p_t_holm"] = A._holm(t2.loc[m, "p_t"].to_numpy())
    display(t2[(t2.scope == "overall") & (t2.unit == "split")][show + ["p_t_holm"]].round(4))
    display(t2[(t2.scope != "overall") & (t2.metric == "pearson_delta") & (t2.unit == "split")][show].round(4))

In [ ]:
# 12. Table 3: per-perturbation paired analysis (bootstrap resamples splits, then pairs within them).
pp = A.per_perturbation(runs)
t3 = pd.concat([A.paired_perturbation(pp, ref, arm, ["pearson_delta", "mse_top20_de", "frac_opposite_top20_de"])
                for ref, arm in COMPARISONS if {ref, arm} <= set(ARMS)], ignore_index=True)
display(t3.round(4))

In [ ]:
# 13. Run-to-run noise: same arm, same split, two training seeds. An effect smaller than this can't be claimed.
w = runs[runs.arm.isin(["GEARS-cs", "VAE-cs"])].pivot_table(index=["arm", "seed"], columns="rep",
                                                             values="test_pearson_delta")
if w.shape[1] >= 2:
    w["abs_diff"] = (w.iloc[:, 0] - w.iloc[:, 1]).abs()
    display(w.round(4)); print(f"mean |difference| between training seeds: {w['abs_diff'].mean():.4f}")
else:
    print("no training-seed replicates yet (tier 6)")

In [ ]:
# 14. Where does it help? Output genes isolated vs connected in GEARS's A0 graph.
gg = A.gene_group_metrics(runs)
t5 = A.gene_group_paired(gg, "GEARS-cs", [a for a in ["VAE-cs", "VAE-random-cs", "GEARS-kNN8-cs", "LASSER-cs"]
                                         if a in ARMS]) if "GEARS-cs" in ARMS else pd.DataFrame()
display(t5.round(4))

In [ ]:
# 15. The VAE graph: stability across splits, overlap with the Pearson kNN graph, consensus edges.
gt = A.graph_table(runs)
display(gt.drop_duplicates(["arm", "seed"]).groupby("arm")[[c for c in ["num_edges", "isolated_genes",
        "degree_median", "h2_hit_added", "h2_hit_random", "h2_hit_a0", "jaccard_to_a0"] if c in gt]]
        .agg(["mean", "std"]).round(3))
consensus = pd.DataFrame()
if "VAE-cs" in ARMS:
    jac, counts = A.edge_stability(runs, "VAE-cs")
    off = jac.to_numpy(dtype=float)[~np.eye(len(jac), dtype=bool)]
    print(f"mean Jaccard of VAE new-edge sets between splits: {np.nanmean(off):.3f}")
    display(A.overlap_with_knn8(runs, "VAE-cs"))
    consensus = A.consensus_edges(runs, "VAE-cs", min_splits=min(3, runs[runs.arm == "VAE-cs"].seed.nunique()))
    display(consensus.head(30))

In [ ]:
# 16. Figures (PDF + PNG).
paths = []
if ARMS:
    paths += F.fig_headline(long, REFS + ARMS, FIG_DIR)
    paths += F.fig_subgroups(long, ARMS, FIG_DIR)
if {"GEARS-cs", "VAE-cs"} <= set(ARMS):
    paths += F.fig_per_perturbation(pp, FIG_DIR, ref="GEARS-cs", arm="VAE-cs")
    paths += F.fig_gene_groups(t5, FIG_DIR, arm="VAE-cs")
    paths += F.fig_graphs(gt, runs, FIG_DIR, a0_arm="GEARS-cs", knn_arm="GEARS-kNN8-cs",
                          lasser_arm="VAE-cs", random_arm="VAE-random-cs")
if len(t2):
    paths += F.fig_forest(t2[t2.unit == "split"], FIG_DIR, "fig_forest_vae", "Paired improvement (splits)")
paths += F.fig_training(A.training_curves(runs), ARMS, FIG_DIR)
from IPython.display import Image, display as _d
for p in paths:
    if p.endswith(".png"):
        _d(Image(p, width=900))

In [ ]:
# 17. Export and zip (download /kaggle/working/lasser_vae_study_results.zip).
A.export_tables({"table1_headline": t1, "table1_subgroups": t1_sub, "table2_paired": t2,
                 "table3_paired_perturbations": t3, "table5_gene_groups": t5, "graph_stats": gt,
                 "consensus_edges": consensus}, RESULTS_DIR)
!cd {STUDY_DIR} && zip -qr /kaggle/working/lasser_vae_study_results.zip results figures runs/runs.csv runs/jobs.json runs/logs
print(sorted(os.listdir(RESULTS_DIR)))

## C. Reading the results

- **The claim** needs `VAE-cs > GEARS-cs` **and** `VAE-cs > VAE-random-cs` (and ideally `> GEARS-kNN8-cs`), with an
  improvement larger than the run-to-run noise in cell 13.
- `VAE-fix` (VAE graph in GEARS's original slot) is expected to match the other "-fix" results of the previous
  study. If it does, that is further support for the re-parameterisation argument in `paper/main.tex`.
- If `GEARS-cs` itself differs strongly from earlier GEARS runs, the cell-signal variant (not the graph) is doing the
  work. Compare only within this study.
- With 3 splits, report effect sizes with CIs, splits-better counts and the per-perturbation analysis; add splits
  4–5 (tier 7) in a later session before drawing conclusions.